### Importation des modules

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

### Chargement des données

In [3]:
# Optionnel mais pratique pour l'affichage dans Jupyter
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# Charger les informations des chiens
df_dogs = pd.read_excel(r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\DogInfo.xlsx")
print(f"Nombre de chiens : {len(df_dogs)}")
display(df_dogs.head())

# Charger les séries temporelles relatives à chaque chien / test
na_values = ['<undefined>']

df_ts = pd.read_csv(
    r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\DogMoveData.csv", 
    na_values=na_values,
    low_memory=False
)

print(f"Dimensions de l'échantillon : {df_ts.shape}")
display(df_ts.head())

Nombre de chiens : 45


,DogID,Breed,Weight,Age months,Gender,NeuteringStatus
0,16,Crossbreed,13,20,1,0
1,18,Belgian Shepherd,29,76,2,0
2,19,German Shepherd,30,57,1,0
3,20,Border Collie,15,47,1,0
4,21,Golden Retriever,39,19,2,0


Dimensions de l'échantillon : (10611068, 20)


,DogID,TestNum,t_sec,ABack_x,ABack_y,ABack_z,ANeck_x,ANeck_y,ANeck_z,GBack_x,GBack_y,GBack_z,GNeck_x,GNeck_y,GNeck_z,Task,Behavior_1,Behavior_2,Behavior_3,PointEvent
0,16,1,0.00,0.041504,0.938965,-0.015137,-0.067871,-0.510254,-0.934570,-17.639161,-22.766115,7.446290,-7.934571,6.347657,13.427735,NaN,NaN,NaN,NaN,NaN
1,16,1,0.01,0.041992,0.941895,-0.020020,-0.128906,-0.494141,-0.913086,-15.075685,-11.413575,4.821778,-3.906250,4.394532,16.540528,NaN,Synchronization,NaN,NaN,NaN
2,16,1,0.02,0.040527,0.939453,-0.004395,-0.158691,-0.480469,-0.911133,-12.207032,-0.122070,2.807617,-0.488281,-1.953125,26.794435,NaN,Synchronization,NaN,NaN,NaN
3,16,1,0.03,0.021484,0.946289,0.007813,-0.122070,-0.486816,-0.880371,-9.460450,7.995606,1.586914,1.159668,-5.676270,38.085940,NaN,Synchronization,NaN,NaN,NaN
4,16,1,0.04,-0.000977,0.951172,0.033691,-0.053711,-0.500000,-0.807129,-8.361817,14.587403,-1.037598,4.577637,4.089356,41.503909,NaN,Synchronization,NaN,NaN,NaN


### Feature engineering : 
- on retire les comportements de "synchronization".
- on créé une nouvelle variable sur la 'taille' du chien (petit, moyen, grand) en fonction de son poids.
- on a déjà retiré les portions de séries temporelles avec valeur de Behavior_1 manquante = pour le train uniquement !.

In [4]:
# 1. Définition des seuils de poids (à ajuster selon ta connaissance vétérinaire si besoin)
# Exemple : < 15 kg (Petit), 15 à 30 kg (Moyen), > 30 kg (Grand)
bins = [0, 15, 30, 100] 
labels = ['Taille_Petite', 'Taille_Moyenne', 'Taille_Grande']

# 2. Création de la catégorie textuelle
df_dogs['Taille_Categorie'] = pd.cut(df_dogs['Weight'], bins=bins, labels=labels, right=False)

# 3. Création des colonnes binaires (0 ou 1) via One-Hot Encoding
df_dogs = pd.get_dummies(df_dogs, columns=['Taille_Categorie'], dtype=int)

# Affichage pour vérifier les nouvelles colonnes
display(df_dogs[['DogID', 'Breed', 'Weight', 'Taille_Categorie_Taille_Petite', 'Taille_Categorie_Taille_Moyenne', 'Taille_Categorie_Taille_Grande']].head())

# Liste des comportements techniques à exclure
comportements_a_ignorer = ['Synchronization', 'Extra_Synchronization']

# Filtrage pour ne conserver que les vrais comportements
df_ts = df_ts[~df_ts['Behavior_1'].isin(comportements_a_ignorer)].copy()

# Vérification rapide
print("Comportements conservés pour la modélisation :")
print(df_ts['Behavior_1'].dropna().unique())

,DogID,Breed,Weight,Taille_Categorie_Taille_Petite,Taille_Categorie_Taille_Moyenne,Taille_Categorie_Taille_Grande
0,16,Crossbreed,13,1,0,0
1,18,Belgian Shepherd,29,0,1,0
2,19,German Shepherd,30,0,0,1
3,20,Border Collie,15,0,1,0
4,21,Golden Retriever,39,0,0,1


Comportements conservés pour la modélisation :
<ArrowStringArray>
[        'Walking',         'Shaking',        'Sniffing',          'Eating',
         'Sitting',        'Trotting',          'Pacing',     'Lying chest',
         'Playing',        'Standing',         'Panting',        'Drinking',
       'Galloping', 'Carrying object',         'Tugging',         'Jumping',
          'Bowing']
Length: 17, dtype: str
